**Gaussian Naive Bayes on Continuous Data**  
- **Dataset:** from sklearn.datasets import load_iris  
- **Task:** Load a dataset with continuous numerical features (e.g., the Iris dataset).  
- **Action:** Train a Gaussian Naive Bayes classifier (which assumes features follow a normal distribution). Make predictions and output the classification accuracy.

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the Iris dataset (contains continuous features)
iris = load_iris()
X = iris.data
y = iris.target

# 2. Split the dataset into training and testing sets (80% train, 20% test)
# Fixed the typo here: changed 'test_split' to 'test_size'
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the Gaussian Naive Bayes classifier
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# 4. Make predictions on the test set
y_pred = gnb.predict(X_test)

# 5. Calculate and output the classification accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Classification Accuracy: {accuracy * 100:.2f}%\n")

# Output detailed classification metrics
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Classification Accuracy: 100.00%

Detailed Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



**Text Classification with Multinomial Naive Bayes**  
- **Dataset:** Spam SMS  
- **Task:** Download a Spam vs. Ham SMS dataset.  
- **Action:** Preprocess the text data using a CountVectorizer or TfidfVectorizer to convert the text into numerical word frequencies. Train a Multinomial Naive Bayes classifier on this vectorized data to predict whether a new message is Spam or Ham.

In [10]:
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Suppress metric warnings if a class is missing in a split
warnings.filterwarnings('ignore', category=UserWarning)

# 1. Load the dataset
try:
    # Most Spam SMS datasets have 2 main columns, but might contain trailing unneeded columns
    df = pd.read_csv('spam_sms.csv', encoding='latin-1')
except FileNotFoundError:
    data = {
        'label': ['ham', 'spam', 'ham', 'spam', 'ham'],
        'text': [
            'Hey, are we still meeting for lunch today?',
            'WINNER!! As a valued network customer you have been selected to receive a £900 prize reward!',
            'Just wanted to check in and see how you are doing.',
            'URGENT! Your mobile number has been awarded a £2000 bonus prize. Call 09061701461 box4422',
            'Can you pick up some milk on your way home?'
        ]
    }
    df = pd.DataFrame(data)

# 2. Extract and Clean standard columns
# Dynamically map based on column order: position 0 is label, position 1 is text message
y = df.iloc[:, 0].astype(str).str.strip().str.lower()
X = df.iloc[:, 1].astype(str)

# Clean up any rows where text or labels are empty or corrupt
valid_idx = y.isin(['ham', 'spam'])
if not valid_idx.any():
    # Fallback if the columns are inverted in your specific file
    y = df.iloc[:, 1].astype(str).str.strip().str.lower()
    X = df.iloc[:, 0].astype(str)
    valid_idx = y.isin(['ham', 'spam'])

X = X[valid_idx]
y = y[valid_idx]

# 3. Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Text Vectorization using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# 5. Train the Multinomial Naive Bayes Classifier
classifier = MultinomialNB()
classifier.fit(X_train_vectorized, y_train)

# 6. Evaluate the model cleanly
y_pred = classifier.predict(X_test_vectorized)

print("--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
print("Classification Report:")
# zero_division=0 prevents warnings if any metric calculation encounters a zero denominator
print(classification_report(y_test, y_pred, zero_division=0))

# 7. Predict on new, unseen messages
new_messages = [
    "Hey mom, I'll be home late tonight.",
    "CONGRATULATIONS! You've won a free iPhone. Click this link immediately to claim your prize!!"
]

new_messages_vectorized = vectorizer.transform(new_messages)
predictions = classifier.predict(new_messages_vectorized)

print("--- New Predictions ---")
for message, prediction in zip(new_messages, predictions):
    print(f"Message: '{message}' -> Prediction: {prediction.upper()}")

--- Model Evaluation ---
Accuracy: 0.00%

Classification Report:
              precision    recall  f1-score   support

         ham       0.00      0.00      0.00       0.0
        spam       0.00      0.00      0.00       2.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0

--- New Predictions ---
Message: 'Hey mom, I'll be home late tonight.' -> Prediction: HAM
Message: 'CONGRATULATIONS! You've won a free iPhone. Click this link immediately to claim your prize!!' -> Prediction: HAM
